# t-SNE Visualization

Companion notebook for Sections 5 and 6 of the lecture notes.

Objectives:

- compare PCA and t-SNE on high-dimensional data;
- inspect the effect of `perplexity`;
- observe the effect of random initialization;
- practice cautious interpretation of t-SNE plots;
- avoid using t-SNE as a standard preprocessing step for supervised models.


In [ ]:
import time
import numpy as np
import pandas as pd
import matplotlib.pyplot as plt

plt.rcParams['figure.figsize'] = (7, 4)
plt.rcParams['axes.grid'] = True


## 1. Dataset and preprocessing

We use a subset of Digits so the notebook runs quickly. Pixel features are standardized before PCA and t-SNE.


In [ ]:
from sklearn.datasets import load_digits
from sklearn.decomposition import PCA
from sklearn.manifold import TSNE, trustworthiness
from sklearn.preprocessing import StandardScaler

X_all, y_all = load_digits(return_X_y=True)
rng = np.random.default_rng(42)
idx = rng.choice(len(X_all), size=500, replace=False)
X = X_all[idx]
y = y_all[idx]
X_scaled = StandardScaler().fit_transform(X)

print('Subset shape:', X_scaled.shape)
print('Classes:', np.unique(y))


## 2. PCA baseline

PCA preserves the best two-dimensional linear variance projection. It is deterministic and fast.


In [ ]:
t0 = time.perf_counter()
X_pca = PCA(n_components=2, random_state=42).fit_transform(X_scaled)
pca_time = time.perf_counter() - t0

fig, ax = plt.subplots(figsize=(7, 6))
scatter = ax.scatter(X_pca[:, 0], X_pca[:, 1], c=y, cmap='tab10', s=14, alpha=0.8)
ax.set_title(f'PCA 2D projection ({pca_time:.2f}s)')
ax.set_xlabel('PC1')
ax.set_ylabel('PC2')
plt.colorbar(scatter, ax=ax, label='digit')
plt.show()

print('PCA trustworthiness:', round(trustworthiness(X_scaled, X_pca, n_neighbors=10), 3))


## 3. t-SNE with a typical perplexity

`t-SNE` builds a probability-based neighborhood description in the original space and seeks a 2D embedding with similar local neighborhoods.


In [ ]:
t0 = time.perf_counter()
X_tsne = TSNE(
    n_components=2,
    perplexity=30,
    init='pca',
    learning_rate='auto',
    max_iter=500,
    random_state=42,
).fit_transform(X_scaled)
tsne_time = time.perf_counter() - t0

fig, ax = plt.subplots(figsize=(7, 6))
scatter = ax.scatter(X_tsne[:, 0], X_tsne[:, 1], c=y, cmap='tab10', s=14, alpha=0.8)
ax.set_title(f't-SNE, perplexity=30 ({tsne_time:.2f}s)')
ax.set_xlabel('t-SNE dimension 1')
ax.set_ylabel('t-SNE dimension 2')
plt.colorbar(scatter, ax=ax, label='digit')
plt.show()

print('t-SNE trustworthiness:', round(trustworthiness(X_scaled, X_tsne, n_neighbors=10), 3))


## 4. Perplexity controls the effective neighborhood size

Low perplexity emphasizes very local structure. Higher perplexity asks each point to consider a broader neighborhood.


In [ ]:
perplexities = [5, 30, 50]
embeddings = {}

fig, axes = plt.subplots(1, len(perplexities), figsize=(15, 4.5))
for ax, perplexity in zip(axes, perplexities):
    emb = TSNE(
        n_components=2,
        perplexity=perplexity,
        init='pca',
        learning_rate='auto',
        max_iter=500,
        random_state=42,
    ).fit_transform(X_scaled)
    embeddings[perplexity] = emb
    score = trustworthiness(X_scaled, emb, n_neighbors=10)
    ax.scatter(emb[:, 0], emb[:, 1], c=y, cmap='tab10', s=10, alpha=0.8)
    ax.set_title(f'perplexity={perplexity}\ntrust={score:.3f}')
    ax.set_xticks([])
    ax.set_yticks([])
plt.tight_layout()
plt.show()


## 5. Random state and layout stability

Different random states can rotate, flip, or rearrange the layout. The exact coordinates are not the scientific object; local neighborhoods and robust visual patterns are.


In [ ]:
states = [0, 7]
fig, axes = plt.subplots(1, 2, figsize=(10, 4.5))
for ax, state in zip(axes, states):
    emb = TSNE(
        n_components=2,
        perplexity=30,
        init='random',
        learning_rate='auto',
        max_iter=500,
        random_state=state,
    ).fit_transform(X_scaled)
    ax.scatter(emb[:, 0], emb[:, 1], c=y, cmap='tab10', s=10, alpha=0.8)
    ax.set_title(f'random_state={state}')
    ax.set_xticks([])
    ax.set_yticks([])
plt.tight_layout()
plt.show()


## 6. Interpretation checklist

Reasonable conclusions from a t-SNE plot:

- some examples have similar local neighborhoods;
- certain labels appear visually mixed or separated;
- the embedding suggests hypotheses worth checking.

Conclusions that require additional evidence:

- faraway clusters are truly far apart in the original space;
- cluster area reflects within-class variance;
- apparent density reflects true data density;
- axes have semantic meaning.


## Exercises

1. Try `perplexity=10` and `perplexity=80`. Which digits become more or less mixed?
2. Replace standardized pixels with PCA to 30 dimensions before t-SNE. Does the plot change?
3. Compute trustworthiness with `n_neighbors=5` and `n_neighbors=30`. How does the score change?


## Takeaways

- t-SNE is mainly an exploratory visualization method.
- Perplexity strongly affects the notion of neighborhood.
- Cluster distances, cluster sizes, densities, and axes should be interpreted cautiously.
